In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/vehicle_sensor_data_cleaned.csv")

df.head()

,vehicle_id,timestamp,engine_temperature,battery_voltage,vibration,brake_temperature,speed,mileage,fault_code,maintenance_required
0,VH002,2026-09-08 09:00:00,107.52,11.90,3.01,87.31,50,33076,F000,Yes
1,VH003,2026-09-08 09:05:00,103.67,11.74,4.70,72.66,57,95784,F001,Yes
2,VH005,2026-09-08 09:10:00,106.75,11.50,4.05,91.00,99,31791,F001,Yes
3,VH001,2026-09-08 09:15:00,92.06,11.96,6.02,67.53,98,23477,F001,No
4,VH002,2026-09-08 09:20:00,96.11,11.76,4.08,88.42,41,80012,F002,No


In [2]:
df.shape

(1000, 10)

In [3]:
df["maintenance_required"].value_counts

<bound method IndexOpsMixin.value_counts of 0      Yes
1      Yes
2      Yes
3       No
4       No
      ... 
995    Yes
996     No
997     No
998     No
999    Yes
Name: maintenance_required, Length: 1000, dtype: str>

In [4]:
x = df[["engine_temperature","battery_voltage","vibration","brake_temperature","speed","mileage","fault_code"]]
y = df["maintenance_required"]

In [5]:
x.head()

,engine_temperature,battery_voltage,vibration,brake_temperature,speed,mileage,fault_code
0,107.52,11.90,3.01,87.31,50,33076,F000
1,103.67,11.74,4.70,72.66,57,95784,F001
2,106.75,11.50,4.05,91.00,99,31791,F001
3,92.06,11.96,6.02,67.53,98,23477,F001
4,96.11,11.76,4.08,88.42,41,80012,F002


In [6]:
y.head()

0    Yes
1    Yes
2    Yes
3     No
4     No
Name: maintenance_required, dtype: str

In [7]:
x.columns

Index(['engine_temperature', 'battery_voltage', 'vibration',
       'brake_temperature', 'speed', 'mileage', 'fault_code'],
      dtype='str')

In [8]:
X = pd.get_dummies(x, columns=["fault_code"], dtype=int)

In [9]:
x.head()

,engine_temperature,battery_voltage,vibration,brake_temperature,speed,mileage,fault_code
0,107.52,11.90,3.01,87.31,50,33076,F000
1,103.67,11.74,4.70,72.66,57,95784,F001
2,106.75,11.50,4.05,91.00,99,31791,F001
3,92.06,11.96,6.02,67.53,98,23477,F001
4,96.11,11.76,4.08,88.42,41,80012,F002


In [10]:
X.shape

(1000, 9)

In [11]:
X.dtypes

engine_temperature    float64
battery_voltage       float64
vibration             float64
brake_temperature     float64
speed                   int64
mileage                 int64
fault_code_F000         int64
fault_code_F001         int64
fault_code_F002         int64
dtype: object

In [12]:
y = y.map({"No": 0, "Yes": 1})

In [13]:
y.head()

0    1
1    1
2    1
3    0
4    0
Name: maintenance_required, dtype: int64

In [14]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import sklearn

print(sklearn.__version__)

1.7.2


In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 9)
X_test: (200, 9)
y_train: (800,)
y_test: (200,)


In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [21]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
y_pred = model.predict(X_test)

In [23]:
y_pred[:10]

array([0, 0, 1, 0, 0, 0, 1, 0, 0, 0])

In [24]:
y_test

791    0
891    0
987    1
449    0
288    0
      ..
257    0
567    0
476    0
562    1
409    0
Name: maintenance_required, Length: 200, dtype: int64

In [25]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(10)

,Actual,Predicted
0,0,0
1,0,0
2,1,1
3,0,0
4,0,0
5,0,0
6,1,1
7,0,0
8,0,0
9,0,0


In [26]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       125
           1       1.00      1.00      1.00        75

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [28]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[125   0]
 [  0  75]]


In [29]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance

,feature,importance
0,engine_temperature,0.259721
3,brake_temperature,0.228102
1,battery_voltage,0.222008
2,vibration,0.214288
5,mileage,0.036488
4,speed,0.029942
6,fault_code_F000,0.003691
7,fault_code_F001,0.003609
8,fault_code_F002,0.002151


In [30]:
import joblib

joblib.dump(model, "../src/ml/vehicle_maintenance_model.pkl")

['../src/ml/vehicle_maintenance_model.pkl']

In [31]:
import os

os.path.exists("../src/ml/vehicle_maintenance_model.pkl")

True